In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
from pathlib import Path

# ─── (0) GLOBAL STYLING VIA rcParams ────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif', 'Serif'],
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 8,
    'ytick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    'axes.linewidth': 1.2,
    'legend.fontsize': 14,
    'legend.frameon': True,
    'legend.framealpha': 0.8,
    'legend.fancybox': True,
    'axes.grid': True,
    'grid.color': '#bbbbbb',
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'savefig.format': 'pdf',
    'savefig.bbox': 'tight',
})

# ─── (1) PATH CONFIGURATION ───────────────────────────────────────────────────
input_root = Path('/Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/results/parametric_convex')
if not input_root.exists():
    raise FileNotFoundError(f"No such folder: {input_root}")

output_root = Path.cwd() / "plots"
output_root.mkdir(parents=True, exist_ok=True)

# ─── (2) PLOTTING PARAMETERS ───────────────────────────────────────────────────
FIGSIZE     = (8, 5)
DPI         = 300
COLORMAP    = plt.get_cmap('Set2')
LINESTYLES  = {'performance': '-', 'guarantee': '--'}
SHOW_INLINE = 0

# ─── (3) UTILITY TO FIND “performance” / “guarantee” COLUMNS ───────────────────
def find_col(columns, *keywords):
    for c in columns:
        low = c.lower()
        if all(kw.lower() in low for kw in keywords):
            return c
    return None

# ─── (4) MAIN LOOP: ENV → PARAM → SEEDS → AGGREGATE & PLOT ─────────────────────
plot_count = 0

for env_dir in sorted(input_root.iterdir()):
    if not env_dir.is_dir():
        continue
    env_name = env_dir.name

    for param_dir in sorted(env_dir.iterdir()):
        if not param_dir.is_dir():
            continue
        param_name = param_dir.name

        seed_dirs = [d for d in sorted(param_dir.iterdir()) if d.is_dir()]
        if not seed_dirs:
            continue

        combos = {}
        for seed_dir in seed_dirs:
            for csv_path in seed_dir.glob(f"{env_name}_*.csv"):
                stem = csv_path.stem
                prefix = env_name + "_"
                algo_name = stem[len(prefix):] if stem.startswith(prefix) else stem
                combos.setdefault(algo_name, []).append(csv_path)

        if not combos:
            continue

        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
        first_algo = True
        is_minimization = False
        legend_handles, legend_labels = [], []   # for legend-only figure

        # collect runtime arrays per algorithm for two extra plots
        runtime_data = []  # list of (algo_name, episodes_x, rt_mean, rt_std)

        for i, algo_name in enumerate(sorted(combos.keys())):
            csv_list = sorted(combos[algo_name])
            if not csv_list:
                continue

            raw_dfs = []
            raw_time = []   # Total Runtime per seed (if available)
            perf_col = guar_col = None
            has_rt = True

            for csv_path in csv_list:
                df = pd.read_csv(csv_path).rename(columns=str.strip)
                if "Episode" not in df.columns:
                    raise KeyError(f"'Episode' column not found in {csv_path.name}")
                df = df.sort_values("Episode").set_index("Episode")

                if perf_col is None and guar_col is None:
                    perf_col = find_col(df.columns, "performance")
                    guar_col = find_col(df.columns, "guarantee")
                    if perf_col is None or guar_col is None:
                        print(f"⚠️  Skipping '{algo_name}' under {env_name}/{param_name}: missing perf/guar columns")
                        perf_col = guar_col = None

                if perf_col is None or guar_col is None:
                    raw_dfs = []
                    break

                raw_dfs.append(df[[perf_col, guar_col]])

                if 'Total Runtime' in df.columns:
                    raw_time.append(df['Total Runtime'])
                else:
                    has_rt = False  # runtime missing for this seed/algorithm

            if not raw_dfs:
                continue

            all_eps = np.sort(np.unique(np.concatenate([df.index.values for df in raw_dfs])))
            perf_mat = np.zeros((len(all_eps), len(raw_dfs)))
            guar_mat = np.zeros_like(perf_mat)

            for col_idx, df_seed in enumerate(raw_dfs):
                seed_x = df_seed.index.values
                perf_mat[:, col_idx] = np.interp(all_eps, seed_x, df_seed[perf_col].values)
                guar_mat[:, col_idx] = np.interp(all_eps, seed_x, df_seed[guar_col].values)

            perf_mean = perf_mat.mean(axis=1)
            perf_std  = perf_mat.std(axis=1) * 0.5
            guar_mean = guar_mat.mean(axis=1)
            guar_std  = guar_mat.std(axis=1)

            # runtime arrays (if present for all seeds)
            if has_rt and len(raw_time) == len(raw_dfs):
                time_mat = np.zeros_like(perf_mat)
                for idx, s_time in enumerate(raw_time):
                    time_mat[:, idx] = np.interp(all_eps, s_time.index.values, s_time.values)
                time_mean = time_mat.mean(axis=1)
                time_std  = time_mat.std(axis=1) * 0.8
                runtime_data.append((algo_name, all_eps, time_mean, time_std))

            if first_algo:
                if perf_mean[-1] < perf_mean[0]:
                    is_minimization = True
                first_algo = False

            color = COLORMAP(i % COLORMAP.N)
            xvals = all_eps

            # Performance
            perf_line, = ax.plot(
                xvals, perf_mean,
                color=color, linestyle=LINESTYLES["performance"],
                linewidth=1.8, label=f"{algo_name} Performance"
            )
            ax.fill_between(xvals, perf_mean - perf_std, perf_mean + perf_std, color=color, alpha=0.25)

            # Guarantee
            guar_line, = ax.plot(
                xvals, guar_mean,
                color=color, linestyle=LINESTYLES["guarantee"],
                linewidth=1.8, label=f"{algo_name} Guarantee"
            )
            ax.fill_between(xvals, guar_mean - guar_std, guar_mean + guar_std, color=color, alpha=0.25)

            # Store handles/labels for legend-only PDF
            legend_handles.extend([perf_line, guar_line])
            legend_labels.extend([f"{algo_name} Performance", f"{algo_name} Guarantee"])

        # ─── axes cosmetics ─────────────────────────────────────────────
        ax.set_xscale("log")
        if is_minimization:
            ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.6)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.6)
        ax.tick_params(axis='both', which='both', labelsize=21, pad=6)
        ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=100))
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2,10)*0.1, numticks=100))
        ax.set_xlabel("Episode", fontsize=28, labelpad=10)
        ax.set_ylabel("Value", fontsize=28, labelpad=10)
        ax.set_title(f"{env_name}  |  {param_name}", fontsize=20, pad=15)

        # ─── (5) SAVE MAIN PLOT WITHOUT LEGEND ───────────────────────────────
        save_dir = output_root / env_name / param_name
        save_dir.mkdir(parents=True, exist_ok=True)
        out_pdf = save_dir / f"{env_name}_{param_name}.pdf"
        fig.savefig(out_pdf, dpi=DPI)
        print(f"✅  Saved plot (no legend): {out_pdf}")
        plt.close(fig)

        # ─── (6) RUNTIME PLOT (vs Episode) — no legend ───────────────────────
        if runtime_data:
            fig_rt, ax_rt = plt.subplots(figsize=FIGSIZE, dpi=DPI)
            for i, (algo_name, x, rt_mean, rt_std) in enumerate(runtime_data):
                color = COLORMAP(i % COLORMAP.N)
                ax_rt.plot(x, rt_mean, color=color, lw=2)
                ax_rt.fill_between(x, rt_mean-rt_std, rt_mean+rt_std, color=color, alpha=0.25)

            ax_rt.set_xscale('log')
            #ax_rt.set_yscale('log')
            ax_rt.minorticks_on()
            ax_rt.grid(which='minor', axis='x', linestyle=':', linewidth=0.5, alpha=0.6)
            ax_rt.spines['top'].set_visible(False)
            ax_rt.spines['right'].set_visible(False)
            ax_rt.tick_params(labelsize=21, pad=6)
            ax_rt.set_xlabel('Episode', fontsize=28, labelpad=5)
            ax_rt.set_ylabel('Total Runtime (s)', fontsize=28, labelpad=5)
            fig_rt.tight_layout()
            out_rt_pdf = save_dir / f"{env_name}_{param_name}_runtime.pdf"
            fig_rt.savefig(out_rt_pdf, dpi=DPI)
            plt.close(fig_rt)
            print(f"✅  Saved runtime plot: {out_rt_pdf}")

            # ─── (7) RUNTIME PLOT (vs Processed Episodes index) — no legend ───
            fig_rtp, ax_rtp = plt.subplots(figsize=FIGSIZE, dpi=DPI)
            for i, (algo_name, x, rt_mean, rt_std) in enumerate(runtime_data):
                color = COLORMAP(i % COLORMAP.N)
                proc_eps = np.arange(1, len(x) + 1)  # processed episodes index
                ax_rtp.plot(proc_eps, rt_mean, color=color, lw=2)
                ax_rtp.fill_between(proc_eps, rt_mean-rt_std, rt_mean+rt_std, color=color, alpha=0.25)

            # x linear (processed count), y log to show scaling
            ax_rtp.set_yscale('log')
            ax_rtp.minorticks_on()
            ax_rtp.grid(which='minor', axis='x', linestyle=':', linewidth=0.5, alpha=0.6)
            ax_rtp.spines['top'].set_visible(False)
            ax_rtp.spines['right'].set_visible(False)
            ax_rtp.tick_params(labelsize=21, pad=6)
            ax_rtp.set_xlabel('Processed Episodes', fontsize=28, labelpad=5)
            ax_rtp.set_ylabel('Total Runtime (s)', fontsize=28, labelpad=5)
            fig_rtp.tight_layout()
            out_rtp_pdf = save_dir / f"{env_name}_{param_name}_runtime_processed.pdf"
            fig_rtp.savefig(out_rtp_pdf, dpi=DPI)
            plt.close(fig_rtp)
            print(f"✅  Saved runtime (processed episodes) plot: {out_rtp_pdf}")

        # ─── (8) CREATE AND SAVE LEGEND AS SEPARATE FIGURE ──────────────────
        fig_leg = plt.figure(figsize=(8, 1.5), dpi=DPI)
        ax_leg = fig_leg.add_subplot(111)
        ax_leg.axis('off')

        legend = ax_leg.legend(
            legend_handles, legend_labels,
            loc='center', ncol=2,
            frameon=True, framealpha=0.85,
            borderpad=0.6, handlelength=1.5, fontsize=14
        )
        legend.get_frame().set_edgecolor('#dddddd')
        legend.get_frame().set_linewidth(0.6)

        out_leg_pdf = save_dir / f"{env_name}_{param_name}_legend.pdf"
        fig_leg.savefig(out_leg_pdf, dpi=DPI, bbox_inches='tight')
        plt.close(fig_leg)
        print(f"✅  Saved legend: {out_leg_pdf}")

        plot_count += 1
        if plot_count <= SHOW_INLINE:
            fig.show()

✅  Saved plot (no legend): /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/BETTING_GAME_CONVEX/n=10,p=0.55/BETTING_GAME_CONVEX_n=10,p=0.55.pdf
✅  Saved runtime plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/BETTING_GAME_CONVEX/n=10,p=0.55/BETTING_GAME_CONVEX_n=10,p=0.55_runtime.pdf
✅  Saved runtime (processed episodes) plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/BETTING_GAME_CONVEX/n=10,p=0.55/BETTING_GAME_CONVEX_n=10,p=0.55_runtime_processed.pdf
✅  Saved legend: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/BETTING_GAME_CONVEX/n=10,p=0.55/BETTING_GAME_CONVEX_n=10,p=0.55_legend.pdf
✅  Saved plot (no legend): /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/BETTING_GAME_CONVEX/n=150,p=0.55/BETTING_GAME_CONVEX_n=150,p=0.55.pdf
✅  Saved runtime plot: /Users/yannik/Documents/Uni/Oxford/PhD/

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
from pathlib import Path

# ─── (0) GLOBAL STYLING VIA rcParams ────────────────────────────────────────
plt.rcParams.update({
    # Serif font (Times New Roman preferred)
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif', 'Serif'],
    # Base font size (will be overridden for labels/ticks below)
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    # Tick styling
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 8,
    'ytick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    # Axis line width
    'axes.linewidth': 1.2,
    # Legend
    'legend.fontsize': 14,
    'legend.frameon': True,
    'legend.framealpha': 0.8,
    'legend.fancybox': True,
    # Grid
    'axes.grid': True,
    'grid.color': '#bbbbbb',
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    # Savefig (explicit DPI in code too)
    'savefig.format': 'pdf',
    'savefig.bbox': 'tight',
    'mathtext.fontset': 'stix',      # STIX looks like Times
    'mathtext.default': 'regular',   # not italic by default
})

# ─── (1) PATH CONFIGURATION ───────────────────────────────────────────────────
input_root = Path('/Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/results/parametric_convex')
if not input_root.exists():
    raise FileNotFoundError(f"No such folder: {input_root}")

output_root = Path.cwd() / "plots"
output_root.mkdir(parents=True, exist_ok=True)

# ─── (2) PLOTTING PARAMETERS ───────────────────────────────────────────────────
FIGSIZE     = (8, 5)            # Rectangular aspect ratio
DPI         = 300               # High DPI for crisp output
COLORMAP    = plt.get_cmap('Set2')  # Muted, pleasant qualitative palette
LINESTYLES  = {'performance': '-', 'guarantee': '--'}
SHOW_INLINE = 0   # We will skip inline display to avoid low-res blurriness

# ─── (3) UTILITY TO FIND “performance” / “guarantee” COLUMNS ───────────────────
def find_col(columns, *keywords):
    """
    Return the first column name that contains ALL of the given keywords (case‐insensitive).
    """
    for c in columns:
        low = c.lower()
        if all(kw.lower() in low for kw in keywords):
            return c
    return None

# ─── (3.5) INFINITY CAPS (tight, sign-aware, small extra space) ───────────────
def apply_infinity_caps(ax, y_min, y_max, has_pos_inf, has_neg_inf, inverted=False, log_scale=True,
                        finite_pad=1.03, cap_pad=1.08):
    """
    Tight limits to finite data + small padding. If ±∞ exist, extend only the
    corresponding side a little more and label that side with ∞ / −∞.
    """
    if log_scale:
        y_min = max(y_min, np.nextafter(0, 1))

        # small breathing room even without infinities
        low_lim  = y_min / finite_pad
        high_lim = y_max * finite_pad

        if has_pos_inf:
            next_dec = 10.0 ** np.ceil(np.log10(y_max))
            high_lim = min(y_max * cap_pad, next_dec / 1.001)
        if has_neg_inf:
            prev_dec = 10.0 ** np.floor(np.log10(y_min))
            low_lim  = max(y_min / cap_pad, prev_dec * 1.001)
    else:
        span = max(y_max - y_min, 1.0)
        low_lim  = y_min - (span * (finite_pad - 1.0))
        high_lim = y_max + (span * (finite_pad - 1.0))
        if has_pos_inf:
            high_lim = y_max + (span * (cap_pad - 1.0))
        if has_neg_inf:
            low_lim  = y_min - (span * (cap_pad - 1.0))

    # Apply limits (respect inversion)
    if not inverted:
        ax.set_ylim(low_lim, high_lim)
    else:
        ax.set_ylim(high_lim, low_lim)

    # Build tick labels with caps as needed (use scientific 10^k on log)
    ticks = [t for t in ax.get_yticks() if (min(low_lim, high_lim) < t < max(low_lim, high_lim))]

    def _fmt(t):
        if log_scale:
            k = np.log10(float(t))
            if abs(k - round(k)) < 1e-9:   # only exact decades -> 10^k
                return rf"$10^{int(round(k))}$"
            return f"{float(t):g}"
        return f"{t:g}"

    labels = [_fmt(t) for t in ticks]
    if has_neg_inf:
        ticks  = [low_lim] + ticks
        labels = ["−∞"] + labels
    if has_pos_inf:
        ticks  = ticks + [high_lim]
        labels = labels + ["∞"]

    ax.set_yticks(ticks)
    ax.set_yticklabels(labels)

# ─── (4) MAIN LOOP: ENV → PARAM → SEEDS → AGGREGATE & PLOT ─────────────────────
plot_count = 0

for env_dir in sorted(input_root.iterdir()):
    if not env_dir.is_dir():
        continue
    env_name = env_dir.name

    for param_dir in sorted(env_dir.iterdir()):
        if not param_dir.is_dir():
            continue
        param_name = param_dir.name

        seed_dirs = [d for d in sorted(param_dir.iterdir()) if d.is_dir()]
        if not seed_dirs:
            continue

        # 4a) Group CSVs by algo: combos[algo_name] = [list of seed CSV paths]
        combos = {}
        for seed_dir in seed_dirs:
            for csv_path in seed_dir.glob(f"{env_name}_*.csv"):
                stem = csv_path.stem
                prefix = env_name + "_"
                algo_name = stem[len(prefix):] if stem.startswith(prefix) else stem
                combos.setdefault(algo_name, []).append(csv_path)

        if not combos:
            continue

        # 4b) Create one high-DPI figure for this (env, param)
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
        first_algo = True
        is_minimization = False

        # Collect for limits and deferred plotting (so we can paint ∞ as part of the line)
        finite_vals_all = []
        series = []  # dict(color, x, perf_mean, perf_std, guar_mean, guar_std, p_pos, p_neg, g_pos, g_neg)

        # 4c) Loop over each algorithm in this env/param
        for i, algo_name in enumerate(sorted(combos.keys())):
            csv_list = sorted(combos[algo_name])
            if not csv_list:
                continue

            # (4c.i) Read seeds into raw_dfs
            raw_dfs = []
            perf_col = guar_col = None

            for csv_path in csv_list:
                df = pd.read_csv(csv_path).rename(columns=str.strip)
                if "Episode" not in df.columns:
                    raise KeyError(f"'Episode' column not found in {csv_path.name}")

                # map textual infinities to ±inf
                df = df.replace(
                    ["Infinity", "Inf", "inf", "+Inf", "-Inf", "+Infinity", "-Infinity"],
                    [np.inf, np.inf, np.inf, np.inf, -np.inf, np.inf, -np.inf],
                )

                df = df.sort_values("Episode").set_index("Episode")

                if perf_col is None and guar_col is None:
                    perf_col = find_col(df.columns, "performance")
                    guar_col = find_col(df.columns, "guarantee")
                    if perf_col is None or guar_col is None:
                        print(f"⚠️  Skipping '{algo_name}' under {env_name}/{param_name}: "
                              f"missing perf/guar columns in {csv_path.name}")
                        perf_col = guar_col = None

                if perf_col is None or guar_col is None:
                    raw_dfs = []
                    break

                raw_dfs.append(df[[perf_col, guar_col]])

            if not raw_dfs:
                continue

            # (4c.ii) Union of episodes
            all_eps = np.sort(np.unique(np.concatenate([df.index.values for df in raw_dfs])))
            num_eps = len(all_eps)
            num_seeds = len(raw_dfs)

            # Matrices for interpolation (NaN -> use nanmean/std)
            perf_mat = np.full((num_eps, num_seeds), np.nan, dtype=float)
            guar_mat = np.full((num_eps, num_seeds), np.nan, dtype=float)

            # Track where any seed had +∞ / −∞
            perf_pos_inf_any = np.zeros(num_eps, dtype=bool)
            perf_neg_inf_any = np.zeros(num_eps, dtype=bool)
            guar_pos_inf_any = np.zeros(num_eps, dtype=bool)
            guar_neg_inf_any = np.zeros(num_eps, dtype=bool)

            # (4c.iii) Interpolate per seed, ignoring non-finite
            for col_idx, df_seed in enumerate(raw_dfs):
                seed_x    = df_seed.index.values.astype(float)
                seed_perf = df_seed[perf_col].to_numpy(dtype=float)
                seed_guar = df_seed[guar_col].to_numpy(dtype=float)

                # record ±∞ by sign
                if np.any(~np.isfinite(seed_perf)):
                    perf_pos_inf_any |= np.isin(all_eps, seed_x[np.isposinf(seed_perf)])
                    perf_neg_inf_any |= np.isin(all_eps, seed_x[np.isneginf(seed_perf)])
                if np.any(~np.isfinite(seed_guar)):
                    guar_pos_inf_any |= np.isin(all_eps, seed_x[np.isposinf(seed_guar)])
                    guar_neg_inf_any |= np.isin(all_eps, seed_x[np.isneginf(seed_guar)])

                finite_p = np.isfinite(seed_perf)
                if finite_p.sum() >= 2:
                    perf_mat[:, col_idx] = np.interp(all_eps, seed_x[finite_p], seed_perf[finite_p], left=np.nan, right=np.nan)

                finite_g = np.isfinite(seed_guar)
                if finite_g.sum() >= 2:
                    guar_mat[:, col_idx] = np.interp(all_eps, seed_x[finite_g], seed_guar[finite_g], left=np.nan, right=np.nan)

            # (4c.iv) Aggregate
            perf_mean = np.nanmean(perf_mat, axis=1)
            perf_std  = np.nanstd (perf_mat, axis=1) * 0.5
            guar_mean = np.nanmean(guar_mat, axis=1)
            guar_std  = np.nanstd (guar_mat, axis=1)

            # collect finite values for tight limits
            if np.isfinite(perf_mean).any():
                finite_vals_all.append(perf_mean[np.isfinite(perf_mean)])
            if np.isfinite(guar_mean).any():
                finite_vals_all.append(guar_mean[np.isfinite(guar_mean)])

            # (4c.v) Minimization detection
            if first_algo:
                idx = np.where(np.isfinite(perf_mean))[0]
                if idx.size >= 2 and perf_mean[idx[-1]] < perf_mean[idx[0]]:
                    is_minimization = True
                first_algo = False

            series.append(dict(
                color=COLORMAP(i % COLORMAP.N),
                x=all_eps,
                perf_mean=perf_mean, perf_std=perf_std,
                guar_mean=guar_mean, guar_std=guar_std,
                p_pos=perf_pos_inf_any, p_neg=perf_neg_inf_any,
                g_pos=guar_pos_inf_any, g_neg=guar_neg_inf_any,
                label_perf=f"{algo_name} Performance",
                label_guar=f"{algo_name} Guarantee",
            ))

        # (5) FINALIZE AXES, GRID, LEGEND, TITLE
        ax.set_xscale("log")
        #ax.set_yscale("log")
        # decade-only majors + standard minors (helps get_yticks() later)
        ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=100))
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2,10)*0.1, numticks=100))
        ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,)))
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10)*0.1))
        if is_minimization:
            ax.invert_yaxis()

        # Prepare markers for ∞ placement
        y_top_marker = None
        y_bottom_marker = None

        # Tighten y strictly to finite data; extend only where ±∞ exist
        has_pos_inf = any(s["p_pos"].any() or s["g_pos"].any() for s in series)
        has_neg_inf = any(s["p_neg"].any() or s["g_neg"].any() for s in series)

        if finite_vals_all:
            finite_vals_all = np.concatenate(finite_vals_all)
            finite_vals_all = finite_vals_all[np.isfinite(finite_vals_all) & (finite_vals_all > 0)]
            if finite_vals_all.size > 0:
                y_min = float(np.nanmin(finite_vals_all))
                y_max = float(np.nanmax(finite_vals_all))
                apply_infinity_caps(
                    ax, y_min, y_max,
                    has_pos_inf=has_pos_inf,
                    has_neg_inf=has_neg_inf,
                    inverted=is_minimization,
                    log_scale=True,
                    finite_pad=1.03,
                    cap_pad=1.08
                )

        # Determine the cap-adjacent y used for plotting ∞ as part of the line
        # (just read back from limits that were set)
        ymin_cur, ymax_cur = ax.get_ylim()
        if not is_minimization:
            y_bottom_marker = ymin_cur
            y_top_marker    = ymax_cur
        else:
            y_top_marker    = ymin_cur
            y_bottom_marker = ymax_cur

        # Now draw the lines; where we saw ±∞, peg to the corresponding cap
        for s in series:
            color = s["color"]; x = s["x"]
            perf_y = s["perf_mean"].copy()
            guar_y = s["guar_mean"].copy()
            if has_pos_inf:
                perf_y[s["p_pos"]] = y_top_marker
                guar_y[s["g_pos"]] = y_top_marker
            if has_neg_inf:
                perf_y[s["p_neg"]] = y_bottom_marker
                guar_y[s["g_neg"]] = y_bottom_marker

            # Lines
            ax.plot(x, perf_y, color=color, linestyle=LINESTYLES["performance"], linewidth=1.8, label=s["label_perf"])
            ax.plot(x, guar_y, color=color, linestyle=LINESTYLES["guarantee"],   linewidth=1.8, label=s["label_guar"])

            # Shaded bands only where finite
            mp = np.isfinite(s["perf_mean"]) & np.isfinite(s["perf_std"])
            if np.any(mp):
                ax.fill_between(x[mp],
                                (s["perf_mean"] - s["perf_std"])[mp],
                                (s["perf_mean"] + s["perf_std"])[mp],
                                color=color, alpha=0.25)
            mg = np.isfinite(s["guar_mean"]) & np.isfinite(s["guar_std"])
            if np.any(mg):
                ax.fill_between(x[mg],
                                (s["guar_mean"] - s["guar_std"])[mg],
                                (s["guar_mean"] + s["guar_std"])[mg],
                                color=color, alpha=0.25)

        # Styling
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.6)
        ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.6)
        ax.tick_params(axis='both', which='both', labelsize=21, pad=6)

        ax.set_xlabel("Episode", fontsize=28, labelpad=10)
        ax.set_ylabel("Value", fontsize=28, labelpad=10)
        ax.set_title(f"{env_name}  |  {param_name}", fontsize=20, pad=15)

        # Legend below the plot
        leg = ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=True,
            framealpha=0.85,
            borderpad=0.6,
            handlelength=1.5,
            fontsize=14
        )
        leg.get_frame().set_edgecolor('#dddddd')
        leg.get_frame().set_linewidth(0.6)

        plt.subplots_adjust(bottom=0.25)
        fig.tight_layout()

        # (6) SAVE
        save_dir = output_root / env_name / param_name
        save_dir.mkdir(parents=True, exist_ok=True)
        out_pdf = save_dir / (f"{env_name}_{param_name}"[0:250] + ".pdf")
        fig.savefig(out_pdf, dpi=DPI)
        print(f"✅  Saved plot: {out_pdf}")

        plt.close(fig)
        plot_count += 1
        if plot_count <= SHOW_INLINE:
            fig.show()

✅  Saved plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/plots/AIRCRAFT_MIXTURE_ONEMOD/theta1=0.369,theta2=0.2,theta3=0.3/AIRCRAFT_MIXTURE_ONEMOD_theta1=0.369,theta2=0.2,theta3=0.3.pdf
✅  Saved plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/plots/AIRCRAFT_MIXTURE_ONEMOD/theta1=0.4,theta2=0.2,theta3=0.15/AIRCRAFT_MIXTURE_ONEMOD_theta1=0.4,theta2=0.2,theta3=0.15.pdf
✅  Saved plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/plots/AIRCRAFT_MIXTURE_ONEMOD_ADAPTIVE/theta1=0.4,theta2=0.2,theta3=0.15/AIRCRAFT_MIXTURE_ONEMOD_ADAPTIVE_theta1=0.4,theta2=0.2,theta3=0.15.pdf
✅  Saved plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/plots/AIRCRAFT_MULTI_SLIP/eps=0.02,r=0.8,p=0.2,maxX=15,maxY=10/AIRCRAFT_MULTI_SLIP_eps=0.02,r=0.8,p=0.2,maxX=15,maxY=10.pdf
✅  Saved plot: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/plots/BETTING_GAME_CONVEX/n=5,p=0.55/B

/var/folders/kd/_30kg3851g7658tq6jwrfn040000gn/T/ipykernel_25550/3781542397.py:245: RuntimeWarning: Mean of empty slice
  perf_mean = np.nanmean(perf_mat, axis=1)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/kd/_30kg3851g7658tq6jwrfn040000gn/T/ipykernel_25550/3781542397.py:247: RuntimeWarning: Mean of empty slice
  guar_mean = np.nanmean(guar_mat, axis=1)
/var/folders/kd/_30kg3851g7658tq6jwrfn040000gn/T/ipykernel_25550/3781542397.py:245: RuntimeWarning: Mean of empty slice
  perf_mean = np.nanmean(perf_mat, axis=1)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/kd/_30kg3851g7658tq6jwrfn040000gn/T/ipykernel_25550/3781542397.py:

RuntimeError: latex was not able to process the following string:
b'\\u221e'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpm5_gf2xj a09baded933cff28885c50a803b2bd66.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./a09baded933cff28885c50a803b2bd66.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(/usr/local/texlive/2025/texmf-dist/tex/latex/psnfss/mathptmx.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/local/texlive/2025/texmf-dist/tex/l

Error in callback <function _draw_all_if_interactive at 0x12ac7d900> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: latex was not able to process the following string:
b'\\u221e'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpqkp09vpn a09baded933cff28885c50a803b2bd66.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./a09baded933cff28885c50a803b2bd66.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(/usr/local/texlive/2025/texmf-dist/tex/latex/psnfss/mathptmx.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/local/texlive/2025/texmf-dist/tex/l

RuntimeError: latex was not able to process the following string:
b'\\u221e'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmp56ft80mw a09baded933cff28885c50a803b2bd66.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./a09baded933cff28885c50a803b2bd66.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/size10.clo))
(/usr/local/texlive/2025/texmf-dist/tex/latex/psnfss/mathptmx.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/local/texlive/2025/texmf-dist/tex/latex/cm-super/type1ec.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/local/texlive/2025/texmf-dist/tex/l

<Figure size 2400x1500 with 1 Axes>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ─── (0) GLOBAL STYLING VIA rcParams ────────────────────────────────────────
plt.rcParams.update({
    # Use LaTeX for all text
    'text.usetex': True,
    # Serif font (Times New Roman preferred)
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif', 'Serif'],
    # Base font size (will be overridden for labels/ticks below)
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    # Tick styling
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 8,
    'ytick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    # Axis line width
    'axes.linewidth': 1.2,
    # Legend
    'legend.fontsize': 14,
    'legend.frameon': True,
    'legend.framealpha': 0.8,
    'legend.fancybox': True,
    # Grid
    'axes.grid': True,
    'grid.color': '#bbbbbb',
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    # Savefig (explicit DPI in code too)
    'savefig.format': 'pdf',
    'savefig.bbox': 'tight',
})

# ─── (1) PATH CONFIGURATION ───────────────────────────────────────────────────
input_root = Path('/Users/yannik/Downloads/plotting/results')
if not input_root.exists():
    raise FileNotFoundError(f"No such folder: {input_root}")

output_root = Path.cwd() / "plots"
output_root.mkdir(parents=True, exist_ok=True)

# ─── (2) PLOTTING PARAMETERS ───────────────────────────────────────────────────
FIGSIZE     = (8, 5)            # Rectangular aspect ratio
DPI         = 300                # High DPI for crisp output
COLORMAP    = plt.get_cmap('Set2')  # Muted, pleasant qualitative palette
LINESTYLES  = {'performance': '-', 'guarantee': '--'}
SHOW_INLINE = 0   # We will skip inline display to avoid low-res blurriness

# ─── (3) UTILITY TO FIND “performance” / “guarantee” COLUMNS ───────────────────
def find_col(columns, *keywords):
    """
    Return the first column name that contains ALL of the given keywords (case‐insensitive).
    """
    for c in columns:
        low = c.lower()
        if all(kw.lower() in low for kw in keywords):
            return c
    return None

# ─── (4) MAIN LOOP: ENV → PARAM → SEEDS → AGGREGATE & PLOT ─────────────────────
plot_count = 0
last_guarantee_data = []

for env_dir in sorted(input_root.iterdir()):
    if not env_dir.is_dir():
        continue
    env_name = env_dir.name

    for param_dir in sorted(env_dir.iterdir()):
        if not param_dir.is_dir():
            continue
        param_name = param_dir.name

        seed_dirs = [d for d in sorted(param_dir.iterdir()) if d.is_dir()]
        if not seed_dirs:
            continue

        # 4a) Group CSVs by algorithm
        combos = {}
        for seed_dir in seed_dirs:
            for csv_path in seed_dir.glob(f"{env_name}_*.csv"):
                stem = csv_path.stem
                prefix = env_name + "_"
                algo = stem[len(prefix):] if stem.startswith(prefix) else stem
                combos.setdefault(algo, []).append(csv_path)

        if not combos:
            continue

        # 4b) Create figure
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
        first_algo = True
        is_minimization = False
        guarantee_data = []

        # 4c) Plot each algorithm
        for i, algo_name in enumerate(sorted(combos.keys())):
            csv_list = sorted(combos[algo_name])
            raw_dfs, perf_col, guar_col = [], None, None

            for csv_path in csv_list:
                df = pd.read_csv(csv_path).rename(columns=str.strip)
                if 'Episode' not in df.columns:
                    raise KeyError(f"'Episode' column not found in {csv_path.name}")
                df = df.sort_values('Episode').set_index('Episode')

                if perf_col is None and guar_col is None:
                    perf_col = find_col(df.columns, 'performance')
                    guar_col = find_col(df.columns, 'guarantee')
                    if perf_col is None or guar_col is None:
                        perf_col = guar_col = None

                if perf_col is None or guar_col is None:
                    raw_dfs = []
                    break

                raw_dfs.append(df[[perf_col, guar_col]])

            if not raw_dfs:
                continue

            all_eps = np.sort(np.unique(np.concatenate([df.index.values for df in raw_dfs])))
            perf_mat = np.zeros((len(all_eps), len(raw_dfs)))
            guar_mat = np.zeros_like(perf_mat)

            for idx, df_seed in enumerate(raw_dfs):
                perf_mat[:, idx] = np.interp(all_eps, df_seed.index.values, df_seed[perf_col])
                guar_mat[:, idx] = np.interp(all_eps, df_seed.index.values, df_seed[guar_col])

            perf_mean, perf_std = perf_mat.mean(1), perf_mat.std(1) * 0.5
            guar_mean, guar_std = guar_mat.mean(1), guar_mat.std(1)
            guarantee_data.append((algo_name, all_eps, guar_mean))

            if first_algo:
                is_minimization = perf_mean[-1] < perf_mean[0]
                first_algo = False

            color = COLORMAP(i % COLORMAP.N)
            # Performance
            ax.plot(all_eps, perf_mean, color=color, linestyle=LINESTYLES['performance'], lw=2,
                    label=f"{algo_name} Performance")
            ax.fill_between(all_eps, perf_mean-perf_std, perf_mean+perf_std, color=color, alpha=0.25)
            # Guarantee
            ax.plot(all_eps, guar_mean, color=color, linestyle=LINESTYLES['guarantee'], lw=2,
                    label=f"{algo_name} Guarantee")
            ax.fill_between(all_eps, guar_mean-guar_std, guar_mean+guar_std, color=color, alpha=0.25)

        # 5) Draw threshold line and intersection dots
        finals = [g[-1] for _, _, g in guarantee_data]
        if finals:
            thr = min(finals)
            eps_min = min(np.min(x) for _, x, _ in guarantee_data)
            eps_max = max(np.max(x) for _, x, _ in guarantee_data)
            ax.hlines(thr, eps_min, eps_max, colors='tomato', linestyles='--', lw=2,
                      label='Min final guarantee')
            for name, xvals, g in guarantee_data:
                mask = (g <= thr) if is_minimization else (g >= thr)
                idx = np.argmax(mask)
                if mask[idx]:
                    ax.plot(xvals[idx], thr, 'o', color='red')

        # 6) Finalize axes and save
        ax.set_xscale('log')
        if is_minimization:
            ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(which='major', linestyle='-', lw=0.7, alpha=0.6)
        ax.grid(which='minor', linestyle=':', lw=0.5, alpha=0.6)
        ax.tick_params(labelsize=21, pad=6)
        ax.set_xlabel('Episode', fontsize=28, labelpad=5)
        ax.set_ylabel('Value', fontsize=28, labelpad=0)
        fig.tight_layout()

        save_dir = output_root / env_name / param_name
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / f"{env_name}_{param_name}.pdf", dpi=DPI)
        plt.close(fig)
        plot_count += 1

        # keep last data for legend
        last_guarantee_data = guarantee_data

# ─── (8) LEGEND-ONLY PLOT (updated snippet) ────────────────────────────────────
if last_guarantee_data:
    # custom order and original Set2 indices
    order = ['McCormick', 'Interval-Arithmetic', 'L1', 'Flat Learning']
    orig_indices = {'McCormick': 2, 'Interval-Arithmetic': 0, 'L1': 1, 'Flat Learning': 3}

    handles, labels = [], []
    for name in order:
        c = COLORMAP(orig_indices[name] % COLORMAP.N)
        handles.append(plt.Line2D([0], [0], color=c, lw=4))
        labels.append(name)

    # gray bars for performance & guarantee
    handles.append(plt.Line2D([0], [0], color='gray', lw=4, linestyle='-'))
    labels.append('Nominal $V_{M}^*$')
    handles.append(plt.Line2D([0], [0], color='gray', lw=4, linestyle='--'))
    labels.append('Robust $V_{\tilde{M}}^*$')

    fig_leg, ax_leg = plt.subplots(figsize=(len(handles)*1.2, 2))
    ax_leg.axis('off')
    leg = ax_leg.legend(handles, labels, ncol=len(handles), frameon=False, loc='center')

    fig_leg.canvas.draw()
    renderer = fig_leg.canvas.get_renderer()
    bbox = leg.get_frame().get_window_extent(renderer)

    # figure dimensions in pixels
    fig_w, fig_h = fig_leg.get_size_inches() * fig_leg.dpi

    # compute x in figure‐coords right after the 4th of 6 entries
    sep_x = (bbox.x0 + (bbox.width * (4.22/6))) / fig_w

    # draw a vertical line in figure coordinates
    fig_leg.add_artist(
        plt.Line2D(
            [sep_x, sep_x], [0.44, 0.56],
            transform=fig_leg.transFigure,
            color='black', lw=1.2
        )
    )

    fig_leg.tight_layout()
    legend_path = output_root / 'legend.pdf'
    fig_leg.savefig(legend_path, dpi=DPI)
    plt.close(fig_leg)
    print(f"✅  Saved legend: {legend_path}")


✅  Saved legend: /Users/yannik/Downloads/plotting/plots/legend.pdf


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ─── (0) GLOBAL STYLING VIA rcParams ────────────────────────────────────────
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif', 'Serif'],
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 8,
    'ytick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    'axes.linewidth': 1.2,
    'legend.fontsize': 14,
    'legend.frameon': True,
    'legend.framealpha': 0.8,
    'legend.fancybox': True,
    'axes.grid': True,
    'grid.color': '#bbbbbb',
    'grid.linestyle': '--',
    'grid.linewidth': 0.8,
    'savefig.format': 'pdf',
    'savefig.bbox': 'tight',
})

# ─── (1) PATH CONFIGURATION ───────────────────────────────────────────────────
input_root = Path('/Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting/results/parametric_convex')
if not input_root.exists():
    raise FileNotFoundError(f"No such folder: {input_root}")

output_root = Path.cwd() / "plots"
output_root.mkdir(parents=True, exist_ok=True)

# ─── (2) PLOTTING PARAMETERS ───────────────────────────────────────────────────
FIGSIZE     = (8, 5)
DPI         = 300
COLORMAP    = plt.get_cmap('Set2')
LINESTYLES  = {'performance': '-', 'guarantee': '--'}

# ─── (3) UTILITY TO FIND “performance” / “guarantee” COLUMNS ───────────────────
def find_col(columns, *keywords):
    for c in columns:
        low = c.lower()
        if all(kw.lower() in low for kw in keywords):
            return c
    return None

# ─── (4) MAIN LOOP: ENV → PARAM → SEEDS → AGGREGATE & PLOT ─────────────────────
plot_count = 0
last_guarantee_data = []

for env_dir in sorted(input_root.iterdir()):
    if not env_dir.is_dir():
        continue
    env_name = env_dir.name

    for param_dir in sorted(env_dir.iterdir()):
        if not param_dir.is_dir():
            continue
        param_name = param_dir.name

        seed_dirs = [d for d in sorted(param_dir.iterdir()) if d.is_dir()]
        if not seed_dirs:
            continue

        # Group CSVs by algorithm
        combos = {}
        for seed_dir in seed_dirs:
            for csv_path in seed_dir.glob(f"{env_name}_*.csv"):
                stem = csv_path.stem
                prefix = env_name + "_"
                algo = stem[len(prefix):] if stem.startswith(prefix) else stem
                combos.setdefault(algo, []).append(csv_path)

        if not combos:
            continue

        # Containers for runtime and guarantee data
        runtime_data = []
        guarantee_data = []

        # Performance & Guarantee plot
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
        first_algo = True
        is_minimization = False

        for i, algo_name in enumerate(sorted(combos.keys())):
            csv_list = sorted(combos[algo_name])
            raw_perf, raw_guar, raw_time = [], [], []
            perf_col = guar_col = None

            # Read per-seed series
            for csv_path in csv_list:
                df = pd.read_csv(csv_path).rename(columns=str.strip)
                df = df.sort_values('Episode').set_index('Episode')

                if perf_col is None and guar_col is None:
                    perf_col = find_col(df.columns, 'performance')
                    guar_col = find_col(df.columns, 'guarantee')

                if perf_col is None or guar_col is None or 'Total Runtime' not in df.columns:
                    raw_perf = raw_guar = raw_time = []
                    break

                raw_perf.append(df[perf_col])
                raw_guar.append(df[guar_col])
                raw_time.append(df['Total Runtime'])

            if not raw_perf:
                continue

            # Align to common episode grid
            all_eps = np.sort(np.unique(np.concatenate([s.index.values for s in raw_perf])))
            perf_mat = np.zeros((len(all_eps), len(raw_perf)))
            guar_mat = np.zeros_like(perf_mat)
            time_mat = np.zeros_like(perf_mat)

            for idx, (s_perf, s_guar, s_time) in enumerate(zip(raw_perf, raw_guar, raw_time)):
                perf_mat[:, idx] = np.interp(all_eps, s_perf.index.values, s_perf.values)
                guar_mat[:, idx] = np.interp(all_eps, s_guar.index.values, s_guar.values)
                time_mat[:, idx] = np.interp(all_eps, s_time.index.values, s_time.values)

            perf_mean, perf_std = perf_mat.mean(1), perf_mat.std(1) * 0.5
            guar_mean, guar_std = guar_mat.mean(1), guar_mat.std(1)
            time_mean, time_std = time_mat.mean(1), time_mat.std(1) * 0.8

            guarantee_data.append((algo_name, all_eps, guar_mean))
            runtime_data.append((algo_name, all_eps, time_mean, time_std))

            if first_algo:
                is_minimization = perf_mean[-1] < perf_mean[0]
                first_algo = False

            color = COLORMAP(i % COLORMAP.N)
            ax.plot(all_eps, perf_mean, color=color, linestyle=LINESTYLES['performance'], lw=2,
                    label=f"{algo_name} Performance")
            ax.fill_between(all_eps, perf_mean-perf_std, perf_mean+perf_std, color=color, alpha=0.25)
            ax.plot(all_eps, guar_mean, color=color, linestyle=LINESTYLES['guarantee'], lw=2,
                    label=f"{algo_name} Guarantee")
            ax.fill_between(all_eps, guar_mean-guar_std, guar_mean+guar_std, color=color, alpha=0.25)

        # Threshold line & markers
        if guarantee_data:
            thr = min(g[-1] for _, _, g in guarantee_data)
            eps_min = min(np.min(x) for _, x, _ in guarantee_data)
            eps_max = max(np.max(x) for _, x, _ in guarantee_data)
            ax.hlines(thr, eps_min, eps_max, colors='tomato', linestyles='--', lw=2,
                      label='Min final guarantee')
            for name, x, g in guarantee_data:
                mask = (g <= thr) if is_minimization else (g >= thr)
                idx = np.argmax(mask)
                if mask[idx]:
                    ax.plot(x[idx], thr, 'o', color='red')

        ax.set_xscale('log')
        #ax.set_yscale('log')
        if is_minimization:
            ax.invert_yaxis()
        
        ax.minorticks_on()
        ax.grid(which='minor', axis='x', linestyle=':', linewidth=0.5, alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=21, pad=6)
        ax.set_xlabel('Episode', fontsize=28, labelpad=5)
        ax.set_ylabel('Value', fontsize=28, labelpad=0)
        fig.tight_layout()

        save_dir = output_root / env_name / param_name
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / f"{env_name}_{param_name}.pdf", dpi=DPI)
        plt.close(fig)
        plot_count += 1

        # ─── (4d) RUNTIME PLOT WITH LOG-LOG & STD BANDS ──────────────────────
        fig_rt, ax_rt = plt.subplots(figsize=FIGSIZE, dpi=DPI)
        for i, (algo_name, x, rt_mean, rt_std) in enumerate(runtime_data):
            color = COLORMAP(i % COLORMAP.N)
            ax_rt.plot(x, rt_mean, color=color, lw=2, label=algo_name)
            ax_rt.fill_between(x, rt_mean-rt_std, rt_mean+rt_std, color=color, alpha=0.25)

        ax_rt.set_xscale('log')
        ax_rt.set_yscale('log')
        ax_rt.minorticks_on()
    # draw minor grid lines only on the x-axis
        ax_rt.grid(which='minor', axis='x', linestyle=':', linewidth=0.5, alpha=0.6)
        ax_rt.spines['top'].set_visible(False)
        ax_rt.spines['right'].set_visible(False)
            # ← NEW: enable minor ticks & draw minor grid
        ax_rt.tick_params(labelsize=21, pad=6)
        ax_rt.set_xlabel('Episode', fontsize=28, labelpad=5)
        ax_rt.set_ylabel('Total Runtime (s)', fontsize=28, labelpad=5)
        fig_rt.tight_layout()
        fig_rt.savefig(save_dir / f"{env_name}_{param_name}_runtime.pdf", dpi=DPI)
        plt.close(fig_rt)

# ─── (8) LEGEND-ONLY PLOT (unchanged) ────────────────────────────────────────
if last_guarantee_data:
    order = ['McCormick', 'Interval-Arithmetic', 'L1', 'Flat Learning']
    orig_indices = {'McCormick': 2, 'Interval-Arithmetic': 0, 'L1': 1, 'Flat Learning': 3}
    handles, labels = [], []
    for name in order:
        c = COLORMAP(orig_indices[name] % COLORMAP.N)
        handles.append(plt.Line2D([0], [0], color=c, lw=4))
        labels.append(name)
    handles.append(plt.Line2D([0], [0], color='gray', lw=4, linestyle='-'))
    labels.append('Nominal $V_{M}^*$')
    handles.append(plt.Line2D([0], [0], color='gray', lw=4, linestyle='--'))
    labels.append('Robust $V_{\tilde{M}}^*$')
    fig_leg, ax_leg = plt.subplots(figsize=(len(handles)*1.2, 2))
    ax_leg.axis('off')
    leg = ax_leg.legend(handles, labels, ncol=len(handles), frameon=False, loc='center')
    fig_leg.canvas.draw()
    renderer = fig_leg.canvas.get_renderer()
    bbox = leg.get_frame().get_window_extent(renderer)
    fig_w, fig_h = fig_leg.get_size_inches() * fig_leg.dpi
    sep_x = (bbox.x0 + (bbox.width * (4.22/6))) / fig_w
    fig_leg.add_artist(
        plt.Line2D(
            [sep_x, sep_x], [0.44, 0.56],
            transform=fig_leg.transFigure, 
            color='black', lw=1.2
        )
    )
    fig_leg.tight_layout()
    legend_path = output_root / 'legend.pdf'
    fig_leg.savefig(legend_path, dpi=DPI)
    plt.close(fig_leg) 
    print(f"✅  Saved legend: {legend_path}")

/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/Users/yannik/miniconda3/envs/quant/lib/python3.10/site-packages/numpy/_core/_methods.py:191: Runtim

In [31]:
import re
from pathlib import Path
import yaml

# ─── CONFIG ──────────────────────────────────────────────────────────────────
INPUT_ROOT = Path('/Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/results/parametric_convex')
OUTPUT_ROOT = Path.cwd() / 'plots'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_TEX = OUTPUT_ROOT / 'benchmark_stats_table.tex'

INCLUDE_PROPERTY = True  # set False to drop the "Property" column
DEBUG_LIST = False       # set True to print a few discovered YAMLs per param

# Preferred benchmark name mapping
BENCHMARK_NAME_MAP = {
    'UAV': 'UAV',
    'UAV_CONVEX': 'UAV',
    'AIRCRAFT_COLLISION': 'Aircraft Collision',
    'AIRCRAFT': 'Aircraft Collision',
    'FIREWIRE': 'Firewire',
    'SEMI_AUTONOMOUS_VEHICLE': 'Semi-Auton. Vehicle',
    'SEMIAUTONOMOUS_VEHICLE': 'Semi-Auton. Vehicle',
    'BETTING_GAME': 'Betting Game',
    'BETTING_GAME_CONVEX': 'Betting Game',
    'BETTING_GAME_CONVEX_ADAPTIVE': 'Betting Game Adaptive',
    'CHAIN': 'Chain',
}

ROW_ORDER = {
    'UAV': 0,
    'Aircraft Collision': 1,
    'Firewire': 2,
    'Semi-Auton. Vehicle': 3,
    'Betting Game': 4,
    'Chain': 5,
}

# ─── HELPERS ─────────────────────────────────────────────────────────────────
def pretty_benchmark_name(env_folder_name: str) -> str:
    key = re.sub(r'[^A-Z_]', '', env_folder_name.upper())
    if env_folder_name in BENCHMARK_NAME_MAP:
        return BENCHMARK_NAME_MAP[env_folder_name]
    if key in BENCHMARK_NAME_MAP:
        return BENCHMARK_NAME_MAP[key]
    for k, v in BENCHMARK_NAME_MAP.items():
        if env_folder_name.upper().startswith(k):
            return v
    return env_folder_name.replace('_', ' ').title()

def safe_load_yaml_info(path: Path):
    """Load YAML and return the ExperimentInfo dict if present, else the dict itself, else None."""
    try:
        with open(path, 'r') as f:
            data = yaml.safe_load(f)
    except Exception as e:
        print(f"⚠️  Failed to load YAML: {path} ({e})")
        return None
    if not isinstance(data, dict):
        return None
    if 'ExperimentInfo' in data and isinstance(data['ExperimentInfo'], dict):
        return data['ExperimentInfo']
    return data

NEEDED_KEYS = {'NumStates', 'NumTransitions', 'NumParameters'}

def pick_base_info(yaml_paths):
    """Choose the first YAML that yields a dict with at least our needed keys."""
    for yp in yaml_paths:
        info = safe_load_yaml_info(yp)
        if isinstance(info, dict) and (NEEDED_KEYS <= set(info.keys())):
            return info, yp
    for yp in yaml_paths:
        info = safe_load_yaml_info(yp)
        if isinstance(info, dict):
            return info, yp
    return None, None

def find_parconvex_yaml(param_dir: Path):
    exact = sorted(param_dir.rglob('*PARCONVEX_FULL_TYING_NOBISIM.yaml'))
    if exact:
        return exact[0]
    any_par = [p for p in param_dir.rglob('*.yaml') if 'PARCONVEX' in p.name.upper()]
    if any_par:
        return sorted(any_par)[0]
    csv_match = sorted(param_dir.rglob('*PARCONVEX_FULL_TYING_NOBISIM.csv'))
    for c in csv_match:
        y = c.with_suffix('.yaml')
        if y.exists():
            return y
    return None

def detect_property(info: dict) -> str:
    t = str(info.get('Type', '')).upper()
    spec = str(info.get('Specification', '')).upper()
    if t.startswith('REW'):
        return '$\\mathbb{E}$'
    import re as _re
    if _re.search(r'\bR(?:MAX|MIN|MAXMIN|MAXMAX)?\s*=\s*\?', spec):
        return '$\\mathbb{E}$'
    return '$\\mathbb{P}$'

def tex_escape(s: str) -> str:
    # Escape LaTeX specials; leave $ as-is for math content in 'Property' (we won't escape Property below)
    return (s.replace('\\', r'\textbackslash{}')
             .replace('&', r'\&')
             .replace('%', r'\%')
             .replace('$', r'\$')
             .replace('#', r'\#')
             .replace('_', r'\_')
             .replace('{', r'\{')
             .replace('}', r'\}')
             .replace('~', r'\textasciitilde{}')
             .replace('^', r'\textasciicircum{}'))

def sort_key(row):
    base = ROW_ORDER.get(row['Benchmark'], 10000)
    return (base, row['Benchmark'], row['Param'])

def _to_int(x, default=10**18):
    """Parse integers safely for sorting (handles commas/whitespace)."""
    try:
        return int(str(x).replace(',', '').strip())
    except Exception:
        return default

# ─── SCAN ────────────────────────────────────────────────────────────────────
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"No such folder: {INPUT_ROOT}")

flat_rows = []  # one per (benchmark, param)
seen = set()

benchmarks = [d for d in sorted(INPUT_ROOT.iterdir()) if d.is_dir()]
if DEBUG_LIST:
    print(f"Found {len(benchmarks)} benchmarks under {INPUT_ROOT}")

for env_dir in benchmarks:
    env_name = env_dir.name
    bench_pretty = pretty_benchmark_name(env_name)

    params = [p for p in sorted(env_dir.iterdir()) if p.is_dir()]
    if DEBUG_LIST:
        print(f"- {env_name}: {len(params)} params")

    for param_dir in params:
        param_name = param_dir.name  # e.g., "n=10,p=0.55" (fallback for ParameterValues)
        key = (env_name, param_name)
        if key in seen:
            continue

        yaml_paths = sorted(param_dir.rglob('*.yaml'))
        if DEBUG_LIST:
            print(f"  · {param_name}: {len(yaml_paths)} yaml files")
            for p in yaml_paths[:3]:
                print(f"    - {p}")

        if not yaml_paths:
            print(f"⚠️  No YAML files under {param_dir}, skipping.")
            continue

        base_info, base_path = pick_base_info(yaml_paths)
        if not isinstance(base_info, dict):
            print(f"⚠️  Could not find a usable YAML with required keys under {param_dir}, skipping.")
            continue

        par_yaml = find_parconvex_yaml(param_dir)
        par_info = safe_load_yaml_info(par_yaml) if par_yaml else None

        num_states = base_info.get('NumStates', '')
        num_trans = base_info.get('NumTransitions', '')
        num_params = base_info.get('NumParameters', '')
        if isinstance(par_info, dict) and 'NumLearnableComponents' in par_info:
            num_expr = par_info['NumLearnableComponents']
        else:
            num_expr = base_info.get('NumLearnableComponents', '')

        prop = detect_property(base_info) if INCLUDE_PROPERTY else ''

        # Parameter values column (prefer YAML, fallback to folder name)
        param_vals = base_info.get('ParameterValues', param_name)

        flat_rows.append({
            'Benchmark': bench_pretty,
            'Param': param_name,         # used for sorting/grouping only
            'ParamVals': param_vals,     # printed column
            'States': num_states,
            'Transitions': num_trans,
            'Parameters': num_params,
            'Expressions': num_expr,
            'Property': prop
        })
        seen.add(key)

# ─── GROUP (for multirow) ────────────────────────────────────────────────────
flat_rows.sort(key=sort_key)
grouped = {}
for r in flat_rows:
    grouped.setdefault(r['Benchmark'], []).append(r)

# Sort rows within each benchmark group by number of states (ascending)
for bench_name, rows in grouped.items():
    rows.sort(key=lambda r: _to_int(r['States']))  # use reverse=True for descending

# ─── RENDER LaTeX (with \multirow in first column) ───────────────────────────
# NOTE: Requires \usepackage{multirow} and \usepackage{array} in your preamble.

if INCLUDE_PROPERTY:
    header_cols = [
        (r'\textbf{Benchmark}',           r'>{\centering\arraybackslash}m{3.8cm}'),
        (r'\textbf{Param.\ Values}',      r'>{\centering\arraybackslash}m{3.0cm}'),
        (r'\textbf{$|\Theta|$}',          r'>{\centering\arraybackslash}m{1cm}'),
        (r'\textbf{$|S|$}',               r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{$|T|$}',               r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{$|\mathcal{E}|$}',     r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{Property}',            r'>{\centering\arraybackslash}m{1.5cm}'),
    ]
    col_keys = ['ParamVals', 'Parameters', 'States', 'Transitions', 'Expressions', 'Property']
else:
    header_cols = [
        (r'\textbf{Benchmark}',           r'>{\centering\arraybackslash}m{4.0cm}'),
        (r'\textbf{Param.\ Values}',      r'>{\centering\arraybackslash}m{3.0cm}'),
        (r'\textbf{\#Parameters}',        r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{\#States}',            r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{\#Transitions}',       r'>{\centering\arraybackslash}m{1.5cm}'),
        (r'\textbf{\#Expressions}',       r'>{\centering\arraybackslash}m{1.5cm}'),
    ]
    col_keys = ['ParamVals', 'Parameters', 'States', 'Transitions', 'Expressions']

col_spec = ' '.join(spec for _, spec in header_cols)
header_line = ' & '.join(h for h, _ in header_cols) + r' \\'

body_lines = []
for bench_name, rows in grouped.items():
    n = len(rows)
    bench_tex = tex_escape(bench_name)
    for idx, r in enumerate(rows):
        # Escape everything except 'Property' which may contain math like $\mathbb{E}$
        vals = []
        for k in col_keys:
            v = '' if r[k] is None else str(r[k])
            vals.append(v if k == 'Property' else tex_escape(v))
        if idx == 0:
            first_cell = rf'\multirow[c]{{{n}}}{{*}}{{{bench_tex}}}'
        else:
            first_cell = ''
        line = ' & '.join([first_cell] + vals) + r' \\'
        body_lines.append(line)

latex = []
latex.append(r'\begin{table}[t]')
latex.append(r'\centering')
latex.append(r'\caption{Salient characteristics of the evaluated benchmarks.}')
latex.append(r'\resizebox{0.93\textwidth}{!}{%')
latex.append(fr'\begin{{tabular}}{{{col_spec}}}')
latex.append(r'\toprule')
latex.append(header_line)
latex.append(r'\midrule')
latex.extend(body_lines)
latex.append(r'\bottomrule')
latex.append(r'\end{tabular}')
latex.append(r'} % end of resizebox')
latex.append(r'\label{tab:stats}')
latex.append(r'\vspace{-7pt}')
latex.append(r'\end{table}')
latex_str = '\n'.join(latex)

with open(OUT_TEX, 'w') as f:
    f.write('% Requires \\usepackage{multirow}\n' + latex_str)

print(f'✅ Wrote LaTeX table to: {OUT_TEX}')
print(f'   Benchmarks: {len(grouped)}  Rows total: {sum(len(v) for v in grouped.values())}')

✅ Wrote LaTeX table to: /Users/yannik/Documents/Uni/Oxford/PhD/prism_convex/prism/prism/plotting_paper/plots/benchmark_stats_table.tex
   Benchmarks: 2  Rows total: 8
